# Uruchamianie eksperymentów

Azure Machine Learning SDK pozwala zlecać i śledzić *zadania* (job), które uruchamiają kod, zapisują metryki i tworzą pliki wyjściowe. To fundament praktycznie każdej pracy z uczeniem maszynowym w Azure Machine Learning.

## Połączenie z obszarem roboczym

Na początek połącz się z obszarem roboczym przy użyciu Azure ML SDK v2.

> **Uwaga**: Jeśli od poprzedniego ćwiczenia wygasła uwierzytelniona sesja z subskrypcją Azure, pojawi się prośba o ponowne zalogowanie.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

print(f"Azure ML gotowe do pracy z obszarem roboczym {ml_client.workspace_name}")

## Śledzenie przebiegu za pomocą MLflow

Podstawowa czynność w pracy z danymi to tworzenie i uruchamianie eksperymentów, które przetwarzają i analizują dane. W tym ćwiczeniu poznasz śledzenie MLflow - wbudowane w Azure Machine Learning - i użyjesz go do uruchomienia kodu Pythona w tym notatniku oraz zapisania wartości wyliczonych z danych. Pracujesz na prostym zbiorze z wynikami badań pacjentów pod kątem cukrzycy. Kod policzy statystyki, narysuje wykres i wybierze próbkę danych. Większość z tego to zwykły Python, taki jak w dowolnej eksploracji danych. Dopiero kilka dodatkowych linii sprawia, że MLflow zapisuje szczegóły tego przebiegu w obszarze roboczym Azure Machine Learning.

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Rozpocznij przebieg MLflow, aby śledzić tę eksplorację
run = mlflow.start_run(run_name="diabetes-exploration")
print("Rozpoczęto przebieg:", run.info.run_id)

# wczytaj dane z pliku lokalnego
data = pd.read_csv('data/diabetes.csv')

# Policz wiersze i zapisz wynik jako metrykę
row_count = (len(data))
mlflow.log_metric('observations', row_count)
print('Analizowanie {} wierszy danych'.format(row_count))

# Narysuj i zapisz wykres liczby pacjentów chorych i zdrowych
diabetic_counts = data['Diabetic'].value_counts()
fig = plt.figure(figsize=(6,6))
ax = fig.gca()
diabetic_counts.plot.bar(ax = ax)
ax.set_title('Pacjenci z cukrzycą')
ax.set_xlabel('Diagnoza')
ax.set_ylabel('Liczba pacjentów')
plt.show()
fig.savefig('label-distribution.png')
mlflow.log_artifact('label-distribution.png')

# zapisz listę występujących liczb ciąż jako parametr
pregnancies = sorted(data.Pregnancies.unique().tolist())
mlflow.log_param('pregnancy_categories', pregnancies)

# Zapisz statystyki opisowe kolumn liczbowych
med_columns = ['PlasmaGlucose', 'DiastolicBloodPressure', 'TricepsThickness', 'SerumInsulin', 'BMI']
summary_stats = data[med_columns].describe().to_dict()
for col in summary_stats:
    for stat, value in summary_stats[col].items():
        mlflow.log_metric(f"{col}_{stat}", value)

# Zapisz próbkę danych i dołącz ją do przebiegu jako artefakt
data.sample(100).to_csv('sample.csv', index=False, header=True)
mlflow.log_artifact('sample.csv')

# Zakończ przebieg
mlflow.end_run()

## Przegląd wyników przebiegu

Gdy przebieg się zakończy, możesz użyć klienta MLflow, aby pobrać informacje o nim i o jego wynikach:

In [ ]:
import json
from mlflow.tracking import MlflowClient

client = MlflowClient()
run_data = client.get_run(run.info.run_id)

# Pobierz zapisane metryki
print("Metryki:")
print(json.dumps(run_data.data.metrics, indent=2))

# Pobierz zapisane parametry
print("\nParametry:")
print(json.dumps(run_data.data.params, indent=2))

# Pobierz artefakty wyjściowe
print("\nArtefakty:")
for artifact in client.list_artifacts(run.info.run_id):
    print(artifact.path)

Postęp zadania i zapisane przez nie metryki możesz śledzić także bezpośrednio w Azure Machine Learning studio.

Otwórz [Azure Machine Learning studio](https://ml.azure.com), wybierz **Jobs** i odszukaj przebieg w eksperymencie **diabetes-experiment**. Oglądając go w studio, zwróć uwagę na karty:

- **Overview** - ogólne właściwości przebiegu.
- **Metrics** - pozwala wybrać zapisane metryki i obejrzeć je w formie tabel albo wykresów.
- **Images** - pokazuje obrazy i wykresy zapisane w przebiegu (tutaj wykres *label-distribution.png*).
- **Outputs + logs** - pliki wyjściowe i artefakty wytworzone przez przebieg.
- **Code** - migawka plików, na podstawie których przebieg powstał.

## Uruchomienie skryptu jako zadania

W poprzednim przykładzie kod wykonywał się bezpośrednio w notatniku. Wygodniejsze rozwiązanie to wydzielić kod do osobnego skryptu, umieścić go w folderze razem z potrzebnymi plikami, a następnie zlecić Azure ML uruchomienie go jako **zadania typu command** na wybranym środowisku obliczeniowym. Dzięki temu łatwo przechodzi się od doraźnej eksploracji na instancji obliczeniowej do powtarzalnych i skalowalnych uruchomień na klastrze obliczeniowym.

Najpierw utwórz folder na pliki skryptu i skopiuj do niego dane:

In [ ]:
import os, shutil

# Utwórz folder na pliki eksperymentu
folder_name = 'diabetes-experiment-files'
experiment_folder = './' + folder_name
os.makedirs(folder_name, exist_ok=True)

# Skopiuj plik z danymi do podfolderu data, aby skrypt wczytywał go
# tą samą ścieżką względną, której używa w tym repozytorium
os.makedirs(os.path.join(folder_name, 'data'), exist_ok=True)
shutil.copy('data/diabetes.csv', os.path.join(folder_name, 'data', 'diabetes.csv'))


Teraz utwórz skrypt Pythona z kodem zadania i zapisz go w tym folderze.

> **Uwaga**: uruchomienie poniższej komórki tylko *tworzy* plik skryptu - nie uruchamia go!

In [ ]:
%%writefile $folder_name/diabetes_experiment.py
import pandas as pd
import os
import mlflow

# wczytaj zbiór danych o cukrzycy
data = pd.read_csv('data/diabetes.csv')

# Policz wiersze i zapisz wynik jako metrykę
row_count = (len(data))
mlflow.log_metric('observations', row_count)
print('Analizowanie {} wierszy danych'.format(row_count))

# Policz i zapisz liczebność poszczególnych etykiet
diabetic_counts = data['Diabetic'].value_counts()
print(diabetic_counts)
for k, v in diabetic_counts.items():
    mlflow.log_metric('Label:' + str(k), v)

# Zapisz próbkę danych w folderze outputs (wyniki zadania są przechwytywane automatycznie)
os.makedirs('outputs', exist_ok=True)
data.sample(100).to_csv("outputs/sample.csv", index=False, header=True)


Zwróć uwagę na kilka rzeczy w tym skrypcie:
- Metryki zapisuje przez `mlflow.log_metric()`. Azure Machine Learning automatycznie podpina przebieg MLflow do każdego zadania, więc wszystko zapisane w ten sposób trafia do historii zadania.
- Dane o cukrzycy wczytuje z podfolderu **data** leżącego obok skryptu. Cała zawartość folderu z kodem jest wysyłana razem z zadaniem, więc po uruchomieniu w chmurze skrypt znajdzie plik pod tą samą ścieżką względną.
- Tworzy folder **outputs** i zapisuje w nim plik z próbką danych - pliki zapisane w **outputs** są automatycznie dołączane do wyników zadania.


Do uruchomienia skryptu jako zadania brakuje już tylko kilku ustaleń:

1. **Środowisko**, w którym skrypt ma działać - tutaj skorzystasz z gotowego (ang. *curated*) środowiska Azure Machine Learning, które zawiera już popularne pakiety Pythona.
2. **Środowisko obliczeniowe**, na którym zadanie ma się wykonać - tutaj klaster `aml-cluster` utworzony we wcześniejszym ćwiczeniu.
3. **Polecenie**, które opisuje, jak uruchomić skrypt.

> **Uwaga**: Konfiguracją środowiska nie przejmuj się na razie zanadto - wrócimy do niej dokładniej w dalszej części kursu.

Poniższa komórka konfiguruje zadanie `command`, a następnie je zleca.

In [ ]:
from azure.ai.ml import command

# skonfiguruj zadanie
job = command(
    code=experiment_folder,
    command="python diabetes_experiment.py",
    environment="azureml://registries/azureml/environments/sklearn-1.5/labels/latest",
    compute="aml-cluster",
    display_name="diabetes-experiment",
    experiment_name="diabetes-experiment",
)

# zleć zadanie
returned_job = ml_client.jobs.create_or_update(job)

# wyświetlaj na bieżąco logi zadania w trakcie jego działania
ml_client.jobs.stream(returned_job.name)

Tak jak poprzednio, wyniki zadania obejrzysz w [Azure Machine Learning studio](https://ml.azure.com), ale możesz też pobrać jego metryki i pliki z poziomu kodu:

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
job_run = client.get_run(returned_job.name)

# Pobierz zapisane metryki
print("Metryki:")
for key, value in job_run.data.metrics.items():
    print(key, value)

print("\nPliki wyjściowe:")
for artifact in client.list_artifacts(returned_job.name):
    print(artifact.path)

## Przegląd historii zadań

Ten sam skrypt został uruchomiony już kilka razy, więc możesz obejrzeć historię w [Azure Machine Learning studio](https://ml.azure.com) i przejrzeć poszczególne zadania. Możesz też wypisać zadania z poziomu SDK i odfiltrować je po nazwie eksperymentu:

In [ ]:
for logged_job in ml_client.jobs.list():
    if logged_job.experiment_name == "diabetes-experiment":
        print('Nazwa zadania:', logged_job.name)
        print('Status:', logged_job.status)
        print('Nazwa wyświetlana:', logged_job.display_name)

> **Więcej informacji**: O uruchamianiu zadań przeczytasz w artykule [Train models with Azure Machine Learning CLI, SDK, and REST API](https://learn.microsoft.com/azure/machine-learning/how-to-train-model) w dokumentacji Azure ML. Szczegóły zapisywania metryk przez MLflow opisuje [Log metrics, parameters, and files with MLflow](https://learn.microsoft.com/azure/machine-learning/how-to-log-view-metrics).